In [ ]:
#| default_exp metre

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import re, unicodedata
from enum import Enum
from fastcore.all import AttrDict, L
from litesearch.sanskrit import strip_vedic, CITE_RE, DEVANAGARI, DEVA_DIGITS, VIRAMA
from ganapati.text import verse_spans

## Three steps

`deva2iast` writes the text in Latin letters keeping vowel length. `syllables` marks each syllable
heavy or light and `scan` returns that as a `g`/`l` string. `detect_meter` matches it against the
tables. A syllable is heavy when its vowel is long or a consonant cluster follows.

`fold_token` throws vowel length away so a search matches either spelling. Metre needs length, so
`deva2iast` is a second transliteration rather than a reuse of the fold.

In [ ]:
#| export
_IV = {'अ':'a','आ':'ā','इ':'i','ई':'ī','उ':'u','ऊ':'ū','ऋ':'ṛ','ॠ':'ṝ','ऌ':'ḷ','ॡ':'ḹ',
       'ए':'e','ऐ':'ai','ओ':'o','औ':'au','ऍ':'e','ऎ':'e','ऑ':'o','ऒ':'o'}
_MV = {'ा':'ā','ि':'i','ी':'ī','ु':'u','ू':'ū','ृ':'ṛ','ॄ':'ṝ','ॢ':'ḷ','ॣ':'ḹ',
       'े':'e','ै':'ai','ो':'o','ौ':'au','ॅ':'e','ॆ':'e','ॉ':'o','ॊ':'o'}
_CI = {'क':'k','ख':'kh','ग':'g','घ':'gh','ङ':'ṅ','च':'c','छ':'ch','ज':'j','झ':'jh','ञ':'ñ',
       'ट':'ṭ','ठ':'ṭh','ड':'ḍ','ढ':'ḍh','ण':'ṇ','त':'t','थ':'th','द':'d','ध':'dh','न':'n',
       'प':'p','फ':'ph','ब':'b','भ':'bh','म':'m','य':'y','र':'r','ल':'l','व':'v',
       'श':'ś','ष':'ṣ','स':'s','ह':'h','ळ':'ḻ','ऴ':'ḻ','ऱ':'r','ऩ':'n',
       'क़':'k','ख़':'kh','ग़':'g','ज़':'j','ड़':'ḍ','ढ़':'ḍh','फ़':'ph','य़':'y'}
_SI = {'ं':'ṃ','ः':'ḥ','ँ':'ṃ','ऽ':'','़':'','ॐ':'oṃ'}

def deva2iast(s:str) -> str:
    'Devanagari to IAST, keeping the vowel length that scansion depends on.'
    s = strip_vedic(s or '')
    out, i, n = [], 0, len(s)
    while i < n:
        ch = s[i]
        if ch in _CI:
            out.append(_CI[ch]); i += 1
            while i < n and s[i] == '़': i += 1
            if i < n and s[i] == VIRAMA: i += 1; continue           # bare consonant, no vowel
            if i < n and s[i] in _MV: out.append(_MV[s[i]]); i += 1; continue
            out.append('a'); continue                                # the implicit vowel
        if ch in _IV: out.append(_IV[ch]); i += 1; continue
        if ch in _SI: out.append(_SI[ch]); i += 1; continue
        if ch in _MV: out.append(_MV[ch]); i += 1; continue           # orphan mātrā
        if ch in DEVA_DIGITS: out.append(' '); i += 1; continue           # a verse number is not a syllable
        if ch == VIRAMA: i += 1; continue
        out.append(ch); i += 1
    return ''.join(out)

In [ ]:
from ganapati.metre import deva2iast

assert deva2iast('श्रीमाता') == 'śrīmātā'
assert deva2iast('देवी') == 'devī'
assert deva2iast('कर्म') == 'karma'      # the virāma kills the implicit vowel
assert deva2iast('गम्') == 'gam'
assert deva2iast('दे॒वी') == 'devī'      # the Vedic tone mark is stripped first

`ai` and `au` are one vowel each, `kh` and `th` one consonant each, so `atha` is two syllables.
A consonant closing one syllable does not open the next: in `kaṃsa` the `ṃ` belongs to `kaṃ`.

In [ ]:
#| export
_VOW2 = ('ai', 'au')
_VOW1 = frozenset('aāiīuūṛṝḷḹeo')
_LONG = frozenset(('ā','ī','ū','ṝ','ḹ','e','o','ai','au'))    # `e` and `o` are always long
_ASP  = ('kh','gh','ch','jh','ṭh','ḍh','th','dh','ph','bh')
_CON1 = frozenset('kgṅcjñṭḍṇtdnpbmyrlvśṣshḻ')
_CODA = frozenset('ṃṁḥ')
_WORDS = re.compile(r'[^\W\d_]+')

def _phones(text:str) -> list:
    "`[('V'|'C', unit)]` for a Sanskrit string, matched word by word so a vowel never fuses across a gap."
    s = deva2iast(text) if DEVANAGARI.search(text or '') else (text or '')
    out = []
    for w in _WORDS.findall(s.lower()):
        i, n = 0, len(w)
        while i < n:
            if w[i:i+2] in _VOW2: out.append(('V', w[i:i+2])); i += 2; continue
            if w[i:i+2] in _ASP:  out.append(('C', w[i:i+2])); i += 2; continue
            # `ḷ` before a vowel is the retroflex lateral, not vocalic l
            if w[i] == 'ḷ' and i+1 < n and (w[i+1:i+3] in _VOW2 or w[i+1] in _VOW1):
                out.append(('C', w[i])); i += 1; continue
            if w[i] in _VOW1:     out.append(('V', w[i]));     i += 1; continue
            if w[i] in _CON1 or w[i] in _CODA: out.append(('C', w[i])); i += 1; continue
            i += 1                                          # not a phoneme; skip it
    return out

def syllables(text:str) -> L:
    'The syllables of a Sanskrit string as `(syllable, guru)` pairs.'
    ph = _phones(text)
    out, onset, coda_at = L(), '', -1
    for j, (k, u) in enumerate(ph):
        # a phone spent as the previous coda is not also this onset (`kaṃsa`, `duḥkha`)
        if k == 'C':
            if j != coda_at: onset += u
            continue
        run, closes = [], True
        for k2, u2 in ph[j+1:]:
            if k2 == 'V': closes = False; break
            run.append(u2)
        heavy = (u in _LONG or any(c in _CODA for c in run) or len(run) >= 2
                 or (closes and len(run) >= 1))
        # at the end of the text the whole run closes the syllable: `gam`, `varam`
        cod = ''.join(run) if closes else ''.join(c for c in run[:1] if c in _CODA)
        if cod: coda_at = j + 1
        out.append((onset + u + cod, heavy))
        onset = ''
    return out

def scan(text:str) -> str:
    'The guru/laghu skeleton of a string as `g`/`l` — what every metre is matched against.'
    return ''.join('g' if h else 'l' for _, h in syllables(text))

In [ ]:
from ganapati.metre import syllables, scan

assert [s for s, _ in syllables('atha')] == ['a', 'tha']      # `th` is one consonant
assert [s for s, _ in syllables('kaṃsa')] == ['kaṃ', 'sa']    # the ṃ closes the first, not opens the second
assert scan('gam') == 'g'                                     # a final cluster makes it heavy
assert scan('rāma') == 'gl'                                   # long ā, then a bare short a
assert scan('īśā vāsyam idaṃ sarvaṃ') == 'gggllggg'

## The gaṇas

A gaṇa is a named triple of weights. Eight cover every triple; `ga` and `la` fill a leftover one
or two. `METERS` stores each metre as a recipe of gaṇa names, the way the tradition states them,
and `ganas` reads a pattern back into names. That is what `by_meter` searches on.

The āryā family counts morae instead: a light syllable is one, a heavy two, built from four-mora
gaṇas with one position that shrinks. `detect_matra_meter` fits weights into slots rather than
matching a fixed string.

In [ ]:
#| export
GANAS = {'ya':'lgg', 'ma':'ggg', 'ta':'ggl', 'ra':'glg',
         'ja':'lgl', 'bha':'gll', 'na':'lll', 'sa':'llg',
         'ga':'g',   'la':'l'}
_OF_GANA = {v: k for k, v in GANAS.items() if len(v) == 3}

def ganas(pattern:str) -> L:
    'The gaṇa names of a `g`/`l` pattern, three at a time; a trailing one or two become `ga`/`la`.'
    s, out = pattern or '', L()
    cut = len(s) - len(s) % 3
    out += L(_OF_GANA[s[i:i+3]] for i in range(0, cut, 3))
    return out + L('ga' if c == 'g' else 'la' for c in s[cut:])

def gana_pattern(recipe:str) -> str:
    'The `g`/`l` pattern a gaṇa recipe spells out: `ta ta ja ga ga` -> `gglgglglgg`.'
    return ''.join(GANAS[x] for x in (recipe or '').split())

METERS = {
    'śālinī':           'ma ta ta ga ga',
    'indravajrā':       'ta ta ja ga ga',
    'upendravajrā':     'ja ta ja ga ga',
    'rathoddhatā':      'ra na ra la ga',
    'svāgatā':          'ra na bha ga ga',
    'vaṃśastha':        'ja ta ja ra',
    'indravaṃśā':       'ta ta ja ra',
    'drutavilambita':   'na bha bha ra',
    'toṭaka':           'sa sa sa sa',
    'bhujaṅgaprayāta':  'ya ya ya ya',
    'praharṣiṇī':       'ma na ja ra ga',
    'rucirā':           'ja bha sa ja ga',
    'vasantatilakā':    'ta bha ja ja ga ga',
    'mālinī':           'na na ma ya ya',
    'pṛthvī':           'ja sa ja sa ya la ga',
    'mandākrāntā':      'ma bha na ta ta ga ga',
    'śikhariṇī':        'ya ma na sa bha la ga',
    'hariṇī':           'na sa ma ra sa la ga',
    'śārdūlavikrīḍita': 'ma sa ja sa ta ta ga',
    'sragdharā':        'ma ra bha na ya ya ya',
}
VRTTA_EXTRA = """
    paṅkti:gllgg tanumadhyā:ggllgg vasumatī:gglllg śaśivadanā:llllgg haṃsamālā:llgglgg kumāralalitā:lglllgg
    madalekhā:gggllgg madhumatī:llllllg citrapadā:gllgllgg haṃsaruta:ggglllgg māṇavaka:gllggllg
    nārācaka:gglglglg pramāṇikā:lglglglg samānikā:glglglgl vidyunmālā:gggggggg bhujagaśiśubhṛtā:llllllggg
    halamukhī:glglllllg campakamālā:gllgggllgg manoramā:lllglglglg mattā:ggggllllgg mayūrasāriṇī:glglglglgg
    paṇava:gggllllggg upasthitā:ggllgllglg śuddhavirāṭ:gggllglglg bhadrikā:llllllglglg
    bhramaravilasitaṃ:ggggllllllg dodhaka:gllgllgllgg mauktikamālā:gllgllggllg sumukhī:llllgllgllg
    sāndrapada:gllggllllgg upasthita:lglllggglgg vātormī:ggggllgglgg vṛntā:llllllllggg śyenikā:glglglglglg
    candravartma:glglllgllllg jaladharamālā:ggggllllgggg jaloddhatagati:lglllglglllg
    kusumavicitrā:llllggllllgg lalitā:gglglllglglg mauktidadāma:lgllgllgllgl maṇimālā:ggllggggllgg
    mālatī:llllgllglglg navamālinī:llllglglllgg pañcacāmara:lglglglglglg pramitākṣarā:llglglllgllg
    pramuditavadanā:llllllglgglg priyaṃvadā:lllglllglglg puṭa:llllllggglgg sragviṇī:glgglgglgglg
    tāmarasa:llllgllgllgg ujjvalā:llllllgllglg vaiśvadevī:gggggglgglgg candrikā:llllllgglglgg
    kṣamā:llllllgglgglg mattamayūra:gggggllggllgg mañjubhāṣiṇī:llglglllglglg nandinī:llglglllgllgg
    alolā:gggllgggggllgg aparājitā:llllllglgllglg asambādhā:gggggllllllggg induvadanā:glllglllglllgg
    praharaṇakalikā:llllllgllllllg candralekhā:gggglgggglgglgg elā:llglgllllllllgg prabhadraka:llllglglllglglg
    śaśikalā:llllllllllllllg vāṇinī:llllglglllglglgg ṛṣabhagajavilasita:gllglglllllllllg
    narkuṭaka:llllglglllgllgllg vaṃśapatrapatitam:gllglglllgllllllg kusumitalatāvellitā:ggggglllllgglgglgg
    meghavisphurjita:lggggglllllgglgglgg suvadanā:gggglggllllllggglllg vṛtta:glglglglglglglglglgl
    madraka:gllglglllglglllglglllg aśvalalita:llllglglllglglllglglllg mattākrīḍa:ggggggggllllllllllllllg
    tanvī:gllggllllllggllgllllllgg krauñcapadā:gllgggllggllllllllllllllg apavāha:gggllllllllllllllllllllggg
    bhujaṅgavijṛmbhita:ggggggggllllllllllglgllglg
"""

_PAT = {k: gana_pattern(v) for k, v in METERS.items()}
_PAT.update({n: q for n, q in (e.split(':') for e in VRTTA_EXTRA.split()) if len(set(q)) > 1})
# upajāti is not a metre but a licence: any mix of these two across the four pādas of one verse.
_MIXED = {'upajāti': ('indravajrā', 'upendravajrā')}

def _pada_ok(w, pat) -> bool:
    'Does one pāda of weights fit a pattern? The last syllable of a pāda is anceps, always.'
    return len(w) == len(pat) and all(x == (c == 'g') for x, c in zip(w[:-1], pat[:-1]))

_PATHYA = 'lgg'
_VIPULA = {'lll':'na-vipulā', 'gll':'bha-vipulā', 'ggg':'ma-vipulā', 'glg':'ra-vipulā'}

def _anustubh(qs):
    'The śloka variant when four 8-syllable pādas scan as one, else None.'
    if not all(list(q[4:7]) == [False, True, False] for q in qs[1::2]): return None
    var = set()
    for q in qs[0::2]:
        k = ''.join('g' if x else 'l' for x in q[4:7])
        if k == _PATHYA: var.add('pathyā')
        elif k in _VIPULA: var.add(_VIPULA[k])
        else: return None
    return ' '.join(sorted(var))

_BARENUM = re.compile(r'(?:\|\||//|॥)\s*[\d०-९]+\s*(?:\|\||//|॥)')
_NOSCAN  = re.compile(r'^[ \t]*>[^\n]*'          # a gloss or translation line
                      r'|^[ \t]*#{1,6}[^\n]*'    # a heading
                      r'|\[[^\]\n]*\]', re.M)    # [h: ... :h], [page 3], the [57M] variant number

def metrical_text(s:str) -> str:
    'A verse with everything removed that is printed beside it but does not scan.'
    return _NOSCAN.sub(' ', _BARENUM.sub(' ', CITE_RE.sub(' ', s or '')))

In [ ]:
#| export
def _meter_member(name:str) -> str:
    'The ASCII enum member for a Sanskrit metre name.'
    out = ''.join(c for c in unicodedata.normalize('NFKD', name) if c.isascii() and c.isalnum()).upper()
    return 'UPASTHITAA' if name == 'upasthitā' else out

In [ ]:
#| export
_G, _S = 'gana', 'syl'
_PURVA  = [(_G,4)]*7 + [(_S,0)]                     # 30 morae
_UTTARA = [(_G,4)]*5 + [(_G,1), (_G,4), (_S,0)]     # 27 morae — the 6th gaṇa shrinks to one laghu
_GITI8  = [(_G,4)]*8                                # 32 morae

MATRA_METERS = {
    'āryā':     (_PURVA,  _UTTARA),   # 30 + 27
    'gīti':     (_PURVA,  _PURVA),    # 30 + 30
    'upagīti':  (_UTTARA, _UTTARA),   # 27 + 27
    'udgīti':   (_UTTARA, _PURVA),    # 27 + 30 — the āryā halves the other way round
    'āryāgīti': (_GITI8,  _GITI8),    # 32 + 32
}
CATURMATRA = ('gg', 'gll', 'llg', 'lgl', 'llll')    # the five four-mora gaṇas; `lgl` is the ja-gaṇa

_METER_NAMES = [*_PAT, *_MIXED, 'anuṣṭubh', *MATRA_METERS]
Meter = Enum('Meter', {_meter_member(n): n for n in _METER_NAMES}, type=str, module=__name__)
Meter.__doc__ = 'Sanskrit metre names with ASCII identifiers and traditional string values.'

In [ ]:
#| export
def matras(text:str) -> int:
    'The morae of a string: one for a laghu, two for a guru.'
    return sum(2 if h else 1 for _, h in syllables(text))

In [ ]:
#| export
def _fit_half(w, slots):
    "Fill one half's ladder of gaṇas from a list of weights; the gaṇa patterns, or None."
    i, pats = 0, []
    for kind, want in slots:
        if kind == _S:                              # the closing guru: one syllable, whatever it is
            if i >= len(w): return None
            pats.append('g'); i += 1; continue
        s, j, pat = 0, i, ''
        while j < len(w) and s < want:
            s += 2 if w[j] else 1
            pat += 'g' if w[j] else 'l'
            j += 1
        if s != want: return None                   # overshot: the boundary is mid-syllable
        pats.append(pat); i = j
    return pats if i == len(w) else None

In [ ]:
#| export
def _half_ok(pats, slots) -> bool:
    'The two gaṇa constraints that separate an āryā from any other 57 morae.'
    if any(pats[i] == 'lgl' for i in (0, 2, 4, 6) if i < len(pats)): return False
    six = pats[5] if len(pats) > 5 else ''
    return six == 'l' if slots[5][1] == 1 else six in ('lgl', 'llll')

In [ ]:
#| export
def detect_matra_meter(text:str, weights:list=None):
    'The āryā-family metre of a verse — `AttrDict(name, halves, ganas, matras)` — or None.'
    w = weights if weights is not None else [h for _, h in syllables(metrical_text(text))]
    for k in range(4, len(w)):
        a, b = list(w[:k]), list(w[k:])
        if not b: continue
        a[-1] = b[-1] = True                        # each half closes on an anceps, read as guru
        for nm, (s1, s2) in MATRA_METERS.items():
            p1, p2 = _fit_half(a, s1), _fit_half(b, s2)
            if p1 and p2 and _half_ok(p1, s1) and _half_ok(p2, s2):
                m1 = sum(2 if h else 1 for h in a)
                m2 = sum(2 if h else 1 for h in b)
                return AttrDict(name=Meter(nm), halves=(m1, m2), matras=m1 + m2,
                                ganas=' '.join(p1) + ' | ' + ' '.join(p2))
    return None

In [ ]:
#| export
def detect_meter(text:str):
    'The metre of one verse as `AttrDict(name, variant, syllables, per_pada, ganas, scan, matras)`, or None.'
    w = [h for _, h in syllables(metrical_text(text))]
    n = len(w)
    if n < 8: return None
    sc, mt = ''.join('g' if h else 'l' for h in w), sum(2 if h else 1 for h in w)
    if not n % 4:
        q  = n // 4
        qs = [w[i*q:(i+1)*q] for i in range(4)]
        sig = ' '.join(ganas(sc[:q][:-1] + 'g'))
        mk = lambda nm, var=None: AttrDict(name=Meter(nm), variant=var, syllables=n, per_pada=q,
                                           ganas=sig, scan=sc, matras=mt, halves=None)
        for nm, pat in _PAT.items():
            if len(pat) == q and all(_pada_ok(x, pat) for x in qs): return mk(nm)
        for nm, (a, b) in _MIXED.items():
            pa, pb = _PAT[a], _PAT[b]
            if len(pa) == q and all(_pada_ok(x, pa) or _pada_ok(x, pb) for x in qs): return mk(nm)
        if q == 8 and (v := _anustubh(qs)) is not None: return mk('anuṣṭubh', v)
    # only now the mora-counting family, so a varṇa metre is never relabelled by it
    if (a := detect_matra_meter(text, w)):
        return AttrDict(name=a.name, variant=None, syllables=n, per_pada=None,
                        ganas=a.ganas, scan=sc, matras=a.matras, halves=a.halves)
    if n % 4: return None
    return AttrDict(name=None, variant=None, syllables=n, per_pada=n//4,
                    ganas=' '.join(ganas(sc[:n//4][:-1] + 'g')), scan=sc, matras=mt, halves=None)

In [ ]:
# Meghadūta 1.1, four pādas of seventeen syllables
mand = ('kaścit kāntāvirahaguruṇā svādhikārātpramattaḥ '
        'śāpenāstaṃgamitamahimā varṣabhogyeṇa bhartuḥ '
        'yakṣaścakre janakatanayāsnānapuṇyodakeṣu '
        'snigdhacchāyātaruṣu vasatiṃ rāmagiryāśrameṣu')
m = detect_meter(mand)
assert m.name is Meter.MANDAKRANTA and m.per_pada == 17
assert m.ganas == 'ma bha na ta ta ga ga'

# Gītā 1.1: the anuṣṭubh, and which of its variants
a = detect_meter('dharmakṣetre kurukṣetre samavetā yuyutsavaḥ '
                 'māmakāḥ pāṇḍavāścaiva kimakurvata sañjaya')
assert a.name is Meter.ANUSTUBH and a.variant == 'pathyā' and a.per_pada == 8

assert Meter.VASANTATILAKA.value == 'vasantatilakā'
assert Meter.MANDAKRANTA == 'mandākrāntā'

## Metre as metadata

`verse_meta` scans every verse in a chunk and returns the union as a flat dict. litesearch indexes
`metadata` for FTS beside `content`, so no new column is needed.

In [ ]:
#| export
PADA_RANGE = (4, 26)

def verse_units(text:str) -> L:
    'The individual verses inside one chunk — a chunk may pack several, and metre is per verse.'
    return L(t for _, _, t, _ in verse_spans(text)) or L([(text or '').strip()])

def verse_meta(text:str) -> dict:
    'The metrical facets of one chunk, as the dict that becomes its `metadata` JSON.'
    ms = L(verse_units(text)).map(detect_meter).filter(lambda m: m is not None)
    if not ms: return {}
    out, named = {}, ms.filter(lambda m: m.name)
    if named:
        out['meter'] = ' '.join(dict.fromkeys(named.attrgot('name')))
        if (vs := [v for v in dict.fromkeys(named.attrgot('variant')) if v]): out['variant'] = ' '.join(vs)
    ok = lambda m: bool(m.name) or PADA_RANGE[0] <= (m.per_pada or 0) <= PADA_RANGE[1]
    var, mat = ms.filter(lambda m: m.per_pada and ok(m)), ms.filter(lambda m: m.halves)
    if (gs := dict.fromkeys(m.ganas.replace(' ', '_') for m in var if m.ganas)): out['gana'] = ' '.join(gs)
    if (ps := dict.fromkeys(str(m.per_pada) for m in var)): out['pada'] = ' '.join(ps)
    if (hs := dict.fromkeys('+'.join(map(str, m.halves)) for m in mat)): out['matra'] = ' '.join(hs)
    return out

In [ ]:
meta = verse_meta(mand)
assert meta['meter'] == Meter.MANDAKRANTA
assert meta['pada'] == '17'                        # 17 syllables per quarter
assert meta['gana'] == 'ma_bha_na_ta_ta_ga_ga'     # the recipe, searchable as one token
assert verse_meta('') == {}

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()